# Ejercicio guiado: Docker para servir un modelo de ML

Bienvenid@. En este notebook vas a **poner en práctica los conceptos de Docker** que vimos en `intro-dockers`, pero esta vez con algo más cercano a MLOps: vas a **empaquetar un modelo de Machine Learning dentro de un contenedor** y a hablarle vía HTTP desde este mismo notebook.

**Tiempo estimado:** 45–60 minutos.

**Pre-requisitos:**
- Docker Desktop instalado y corriendo (el ícono de la ballena en la barra debe estar activo).
- Python 3.9+ y `pip` (o `uv`) disponibles.
- Haber leído `README.md` y `COMANDOS_DOCKER.md` de `intro-dockers`.

**Lo que vas a construir:**

```
┌──────────────────────────────────────────────────────────┐
│   Tu navegador / Notebook                                │
│           │                                              │
│           │  HTTP  (requests.post)                       │
│           ▼                                              │
│   localhost:8080  ─────────►   localhost:8000            │
│   (tu computador)              (dentro del contenedor)   │
│                                                          │
│              ┌────────────────────────┐                  │
│              │  Contenedor Docker     │                  │
│              │  ┌──────────────────┐  │                  │
│              │  │  FastAPI (app)   │  │                  │
│              │  │  + modelo Iris   │  │                  │
│              │  │  (sklearn)       │  │                  │
│              │  └──────────────────┘  │                  │
│              └────────────────────────┘                  │
└──────────────────────────────────────────────────────────┘
```

**Cómo usar este notebook:**
1. Lee cada bloque de texto **antes** de ejecutar la celda que le sigue.
2. Las celdas están pensadas para ejecutarse **en orden**.
3. Cuando veas un bloque "Te toca a ti", detente y prueba.
4. Habrá dos entornos mentales: este notebook (para comandos cortos y `requests`) y, sólo en la Parte 2, **una terminal aparte** para correr la app localmente.

---


## Hoja de ruta

| Parte | Qué haces | Por qué importa |
|-------|-----------|-----------------|
| 1 | Entrenas un modelo pequeñito y lo guardas en disco | Un modelo sin forma de servirse no vale nada en producción |
| 2 | Envuelves el modelo en una API (FastAPI) | Así otros sistemas pueden consumirlo |
| 3 | Corres la API **localmente** y le hablas desde el notebook | Confirmas que funciona en tu máquina |
| 4 | **Dockerizas** la API: Dockerfile, imagen, contenedor | Así funciona igual en tu máquina, en la del compañero, en CI, en la nube |
| 5 | Retos: caché, múltiples contenedores, logs, inspeccionar | Afianzas comandos cotidianos de Docker |
| 6 | Limpieza y reflexión | Un buen ingeniero MLOps deja todo ordenado |

## La gran analogía

> Imagina que cocinaste un **pastel delicioso** (el modelo) en tu casa, pero tu tía en otra ciudad quiere el mismo pastel. Puedes:
>
> - Mandarle **la receta** (código): tiene que comprar ingredientes, tener horno, esperar que le salga igual. Frágil.
> - Mandarle **el pastel empacado** en una caja sellada que mantiene temperatura y no se derrama. Eso es **Docker**.
>
> | Concepto Docker | Equivalente culinario |
> |-----------------|------------------------|
> | **Dockerfile** | La lista de instrucciones para armar la caja |
> | **Imagen** | La caja sellada con el pastel adentro, lista para enviar |
> | **Contenedor** | La caja ya abierta sobre la mesa de tu tía, lista para comerse |

Con esto en mente, empezamos.

---


## Parte 0 — Verifica tu entorno

Antes de nada, confirma que Docker y Python funcionan. Si alguna celda falla, **no sigas** hasta arreglarlo.


In [ ]:
# 1. Versión de Docker (cliente). Si falla, Docker no está instalado.
!docker --version

In [ ]:
# 2. Docker realmente corriendo (habla con el demonio). Si falla, abre Docker Desktop.
!docker info --format 'Server version: {{.ServerVersion}}'

In [ ]:
# 3. Python del notebook
import sys
print("Python:", sys.version.split()[0])
assert sys.version_info >= (3, 9), "Necesitas Python 3.9+"
print("OK")

---

## Parte 1 — Entrenar y guardar un modelo pequeñito

Vamos a entrenar un clasificador del dataset **Iris** (tres especies de flores). Es pequeño, entrena en menos de un segundo, y así nos enfocamos en Docker, no en el modelo.

**Objetivo:** tener un archivo `model.pkl` que después la API va a cargar.

> En términos de la analogía: éste es el **pastel**. La Parte 2 y 3 lo meten en la caja.


In [ ]:
# Instala dependencias EN EL NOTEBOOK (no son las mismas que Docker usará dentro del contenedor).
# Si ya las tienes instaladas, no pasa nada.
%pip install -q scikit-learn==1.5.2 joblib==1.4.2 fastapi==0.115.0 "uvicorn[standard]==0.32.0" requests==2.32.3

In [ ]:
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
import joblib

# 1. Cargar datos
data = load_iris()
X, y = data.data, data.target
feature_names = list(data.feature_names)
target_names = list(data.target_names)

# 2. Separar train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Entrenar un modelo simple
model = LogisticRegression(max_iter=200).fit(X_train, y_train)
print("Accuracy en test:", round(model.score(X_test, y_test), 3))

# 4. Guardar el modelo en disco (con sus metadatos)
joblib.dump({"model": model, "feature_names": feature_names, "target_names": target_names}, "model.pkl")
print("Modelo guardado en model.pkl")

In [ ]:
# Probamos cargar el modelo como lo hará la API
bundle = joblib.load("model.pkl")
sample = [[5.1, 3.5, 1.4, 0.2]]  # sepal_length, sepal_width, petal_length, petal_width
pred = bundle["model"].predict(sample)[0]
print("Predicción para", sample[0], "→", bundle["target_names"][pred])

---

## Parte 2 — Envolver el modelo en una API

Un modelo guardado solito no es útil para nadie que no sepa Python. Lo envolvemos en una **API HTTP** con FastAPI para que cualquier sistema pueda preguntarle cosas mandando JSON.

### Archivos que vamos a crear
- `requirements.txt` — las dependencias (para Docker).
- `app.py` — el código de la API.

Usamos `%%writefile` para crearlos **desde este notebook**, sin tener que abrir un editor aparte.


In [ ]:
%%writefile requirements.txt
fastapi==0.115.0
uvicorn[standard]==0.32.0
scikit-learn==1.5.2
joblib==1.4.2
numpy==1.26.4

In [ ]:
%%writefile app.py
"""
API mínima que sirve un modelo Iris con FastAPI.
"""
from fastapi import FastAPI
from pydantic import BaseModel, Field
import joblib
import os
import socket

app = FastAPI(title="Iris API", version="1.0.0")

# Cargamos el modelo UNA sola vez al arrancar (no en cada request → sería lento)
BUNDLE = joblib.load("model.pkl")
MODEL = BUNDLE["model"]
FEATURES = BUNDLE["feature_names"]
CLASSES = BUNDLE["target_names"]


class IrisInput(BaseModel):
    sepal_length: float = Field(..., example=5.1)
    sepal_width: float = Field(..., example=3.5)
    petal_length: float = Field(..., example=1.4)
    petal_width: float = Field(..., example=0.2)


@app.get("/")
def home():
    return {
        "mensaje": "Hola, soy una API que clasifica flores Iris.",
        "endpoints": ["/health", "/info", "/predict (POST)"],
        "host_interno": socket.gethostname(),
    }


@app.get("/health")
def health():
    return {"status": "ok"}


@app.get("/info")
def info():
    return {
        "features": FEATURES,
        "classes": CLASSES,
        "running_in_docker": os.path.exists("/.dockerenv"),
        "hostname": socket.gethostname(),
    }


@app.post("/predict")
def predict(x: IrisInput):
    values = [[x.sepal_length, x.sepal_width, x.petal_length, x.petal_width]]
    pred = int(MODEL.predict(values)[0])
    proba = MODEL.predict_proba(values)[0].tolist()
    return {
        "class_id": pred,
        "class_name": CLASSES[pred],
        "probabilities": dict(zip(CLASSES, proba)),
    }


if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)


### Mini-disección de `app.py`

| Línea(s) | Qué hace | Por qué |
|----------|---------|---------|
| `BUNDLE = joblib.load("model.pkl")` | Carga el modelo **una vez** | Cargar por request sería lento |
| `class IrisInput(BaseModel)` | Valida la entrada con Pydantic | Si alguien manda basura, FastAPI responde `422` automáticamente |
| `GET /health` | Endpoint mínimo de salud | Docker, Kubernetes, etc., lo usan para saber si la app está viva |
| `GET /info` | Reporta si corre en Docker | Vamos a usarlo para distinguir "local" vs "Docker" |
| `POST /predict` | Recibe features, devuelve clase + probabilidades | El corazón de la API |
| `host="0.0.0.0"` | Escucha en **todas** las interfaces | **Indispensable** para que Docker pueda exponer el puerto |

> **Nota clave:** si pusieras `host="127.0.0.1"` dentro del contenedor, Docker no podría exponer el puerto hacia tu máquina. Éste es uno de los errores más comunes al dockerizar.

---

## Correr la app localmente (sin Docker todavía)

Vamos a verificar que la app funciona **antes** de meterla en Docker. Así, si algo falla, sabemos que no es culpa de Docker.

**Abre una terminal nueva** (no en el notebook, una terminal aparte del sistema), ubícate en la carpeta de este notebook, y corre **uno** de estos dos comandos:

```bash
# Con pip tradicional
pip install -r requirements.txt
python app.py
```

```bash
# Con uv (más rápido)
uv run --with-requirements requirements.txt python app.py
```

Deberías ver algo parecido a:

```
INFO:     Started server process [12345]
INFO:     Uvicorn running on http://0.0.0.0:8000
```

**Deja esa terminal corriendo** y vuelve al notebook. La siguiente celda le habla a esa app.


In [ ]:
import requests

BASE = "http://127.0.0.1:8000"
print("GET  /health  →", requests.get(f"{BASE}/health").json())
print("GET  /info    →", requests.get(f"{BASE}/info").json())

payload = {"sepal_length": 5.1, "sepal_width": 3.5, "petal_length": 1.4, "petal_width": 0.2}
print("POST /predict →", requests.post(f"{BASE}/predict", json=payload).json())

### Te toca a ti (antes de seguir)

Con la app local corriendo, **cambia el payload** y prueba con flores distintas:

- Típica **setosa**: `sepal_length=5.1, sepal_width=3.5, petal_length=1.4, petal_width=0.2`
- Típica **versicolor**: `sepal_length=6.0, sepal_width=2.7, petal_length=5.1, petal_width=1.6`
- Típica **virginica**: `sepal_length=6.3, sepal_width=3.3, petal_length=6.0, petal_width=2.5`

Fíjate que `/info` dice `"running_in_docker": false`. **Recuerda este detalle** — cuando corra en Docker cambiará.

Cuando termines, **detén la app local** yendo a la terminal y presionando `Ctrl+C`. Necesitamos el puerto 8000 libre para confiar en que lo que responda después es realmente el contenedor.

---


## Parte 3 — Dockerizar la API

### ¿Qué es Docker? (versión corta)

Docker empaqueta **tu código + Python + todas las librerías + el modelo** en un "cajón aislado" (el contenedor) que corre igual en tu Mac, en el Linux de un servidor, y en el Windows de tu compañero.

```
  Tu máquina (host)
  ┌────────────────────────────────────────┐
  │  Docker engine (demonio)               │
  │  ┌──────────────────────────────────┐  │
  │  │  Imagen: ml-iris-app             │  │
  │  │   - python:3.11-slim             │  │
  │  │   - fastapi, uvicorn, sklearn    │  │
  │  │   - app.py + model.pkl           │  │
  │  └──────────────────────────────────┘  │
  │                │                       │
  │                │  docker run           │
  │                ▼                       │
  │  ┌──────────────────────────────────┐  │
  │  │  Contenedor: mi-iris             │  │
  │  │  proceso uvicorn corriendo       │  │
  │  │  puerto 8000 dentro              │  │
  │  └──────────────────────────────────┘  │
  │                ▲                       │
  │         puerto 8080 → 8000             │
  └────────────────────────────────────────┘
```

### Los tres conceptos que más se confunden

| Concepto | Analogía | Comando típico |
|----------|----------|----------------|
| **Dockerfile** | La receta escrita | `docker build` lo lee |
| **Imagen** | El molde / caja sellada, inmutable | `docker images` para listarlas |
| **Contenedor** | Una instancia en ejecución | `docker ps` para verlos |

> Puedes tener **1 imagen y N contenedores** corriendo al mismo tiempo, todos iguales pero independientes. Lo vamos a demostrar más adelante.

### Archivos que vamos a crear ahora

- `.dockerignore` — qué archivos NO copiar al contenedor (reduce el tamaño).
- `Dockerfile` — la receta para construir la imagen.


In [ ]:
%%writefile .dockerignore
__pycache__
*.pyc
.ipynb_checkpoints
*.ipynb
.venv
.git
.DS_Store
README.md

In [ ]:
%%writefile Dockerfile
# Imagen base oficial de Python (variante "slim" = más liviana)
FROM python:3.11-slim

# Directorio de trabajo dentro del contenedor
WORKDIR /app

# 1. Copiar SOLO requirements.txt primero (truco de cache)
COPY requirements.txt .

# 2. Instalar dependencias (esta capa se cachea si requirements.txt no cambia)
RUN pip install --no-cache-dir -r requirements.txt

# 3. Copiar el resto: código y modelo entrenado
COPY app.py .
COPY model.pkl .

# Variable útil para que los prints aparezcan en los logs al instante
ENV PYTHONUNBUFFERED=1

# Documentar el puerto interno (uvicorn corre en 8000)
EXPOSE 8000

# Comando que se ejecuta al arrancar el contenedor
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]

### El Dockerfile, línea por línea

| Instrucción | Qué hace | Analogía culinaria |
|-------------|---------|-------------------|
| `FROM python:3.11-slim` | Parte de una imagen base con Python ya instalado | "Comprar la base del pastel ya hecha" |
| `WORKDIR /app` | Se posiciona en `/app` dentro del contenedor | "Pararte frente a la mesa de trabajo" |
| `COPY requirements.txt .` | Copia **solo** la lista de dependencias | Esto **primero**, por el caché |
| `RUN pip install ...` | Instala las librerías | La capa más lenta → por eso queremos cachearla |
| `COPY app.py .` y `COPY model.pkl .` | Copia código + modelo | Al final, porque cambian más seguido |
| `EXPOSE 8000` | **Documenta** el puerto (no lo abre) | Una etiqueta: "por aquí salen las respuestas" |
| `CMD [...]` | El comando que se ejecuta al arrancar | "Encender el horno" |

> **¿Por qué `COPY requirements.txt` antes que `COPY app.py`?**
> Docker construye la imagen en **capas**. Si cambias sólo `app.py`, Docker reutiliza la capa cacheada de `pip install` (~30 s) y sólo rehace la capa de copia (~0.1 s). Si copiaras todo junto, **cada cambio de código reinstalaría todo**. Este truco te ahorra horas al cabo del mes.

---

### Construir la imagen

Esto lee el Dockerfile, arma la imagen y la guarda localmente con el tag `ml-iris-app`. La primera vez puede tardar 1–3 minutos porque baja Python y las librerías.


In [ ]:
!docker build -t ml-iris-app .

### Verificar que la imagen existe

Deberías ver una línea con `ml-iris-app`, su tag `latest`, un IMAGE ID, y un tamaño aproximado de ~200–300 MB.


In [ ]:
!docker images ml-iris-app

### Correr el contenedor

Lanzamos la imagen como contenedor en modo **detached** (`-d`, segundo plano) y **mapeamos puertos**: `-p 8080:8000` significa *"lo que llegue al puerto 8080 de mi máquina, mándalo al 8000 del contenedor"*.

> Si en la Parte 2 dejaste la app local corriendo, **párala antes** (Ctrl+C en la terminal) para evitar conflictos de puerto.

> La primera línea borra cualquier contenedor viejo con el mismo nombre — útil si ejecutas este notebook varias veces.


In [ ]:
# Eliminar cualquier contenedor previo con el mismo nombre (si no existe, ignora el error).
!docker rm -f mi-iris 2>/dev/null; true

# Arrancar el contenedor
!docker run -d -p 8080:8000 --name mi-iris ml-iris-app

In [ ]:
!docker ps --filter "name=mi-iris" 

### Hablarle a la API dentro del contenedor

Fíjate: desde el notebook usamos el puerto **8080** (el de tu máquina), pero adentro del contenedor la app sigue corriendo en 8000. Docker hace el puente.


In [ ]:
import time, requests

# Damos 2 segundos a uvicorn para arrancar dentro del contenedor
time.sleep(2)

BASE = "http://127.0.0.1:8080"
print("GET  /health  →", requests.get(f"{BASE}/health").json())
print("GET  /info    →", requests.get(f"{BASE}/info").json())

payload = {"sepal_length": 6.3, "sepal_width": 3.3, "petal_length": 6.0, "petal_width": 2.5}
print("POST /predict →", requests.post(f"{BASE}/predict", json=payload).json())

### Observa esto con cuidado

1. `/info` ahora dice `"running_in_docker": true`. El **mismo código** responde `false` cuando corre local y `true` cuando corre en Docker — porque la función detecta el archivo `/.dockerenv` que sólo existe dentro de contenedores.
2. `hostname` es un ID corto random (algo como `a3f8c9d2e1b4`): es el ID del contenedor.
3. **Mismo modelo, misma API, misma predicción.** Lo único que cambió es el envoltorio.

### Ver los logs del contenedor

Los `print` y los mensajes de uvicorn quedan acá. Es tu principal herramienta de debugging cuando algo no arranca.


In [ ]:
!docker logs mi-iris

---

## Parte 4 — Retos rápidos

Ya tienes todo corriendo. Ahora afianzamos comandos con cuatro retos cortos.

### Reto 1 — El poder del caché

Construye la imagen **otra vez sin cambiar nada**. Cronometra. Vas a ver que todas las capas quedan `CACHED` y termina en unos segundos.


In [ ]:
import time
t0 = time.time()
!docker build -t ml-iris-app .
print(f"\nTiempo total: {time.time()-t0:.1f}s")

Ahora modifica el mensaje del endpoint `/` (solo un texto) para ver qué capas se recachean y cuáles se rehacen. La siguiente celda reescribe `app.py` con un cambio mínimo.


In [ ]:
%%writefile app.py
"""API mínima que sirve un modelo Iris con FastAPI."""
from fastapi import FastAPI
from pydantic import BaseModel, Field
import joblib, os, socket

app = FastAPI(title="Iris API", version="1.0.1")

BUNDLE = joblib.load("model.pkl")
MODEL = BUNDLE["model"]
FEATURES = BUNDLE["feature_names"]
CLASSES = BUNDLE["target_names"]


class IrisInput(BaseModel):
    sepal_length: float = Field(..., example=5.1)
    sepal_width: float = Field(..., example=3.5)
    petal_length: float = Field(..., example=1.4)
    petal_width: float = Field(..., example=0.2)


@app.get("/")
def home():
    return {
        "mensaje": "Soy la versión 1.0.1 — cambie el mensaje para ver el caché en acción.",
        "endpoints": ["/health", "/info", "/predict (POST)"],
        "host_interno": socket.gethostname(),
    }


@app.get("/health")
def health():
    return {"status": "ok", "version": "1.0.1"}


@app.get("/info")
def info():
    return {
        "features": FEATURES,
        "classes": CLASSES,
        "running_in_docker": os.path.exists("/.dockerenv"),
        "hostname": socket.gethostname(),
    }


@app.post("/predict")
def predict(x: IrisInput):
    values = [[x.sepal_length, x.sepal_width, x.petal_length, x.petal_width]]
    pred = int(MODEL.predict(values)[0])
    proba = MODEL.predict_proba(values)[0].tolist()
    return {
        "class_id": pred,
        "class_name": CLASSES[pred],
        "probabilities": dict(zip(CLASSES, proba)),
    }


if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)


In [ ]:
import time
t0 = time.time()
!docker build -t ml-iris-app .
print(f"\nTiempo total: {time.time()-t0:.1f}s")

En el log de esa construcción deberías ver que:

- `COPY requirements.txt` → **CACHED** (no cambió)
- `RUN pip install` → **CACHED** (no cambió; **el paso más lento se salta**)
- `COPY app.py` → se rehace (sí cambió)
- Las capas de abajo se rehacen

> Sin el truco del caché, cada cambio en tu app tomaría ~30 s. Con el truco: ~0.5 s. **Éste es el motivo real por el que el orden de las instrucciones en el Dockerfile importa.**

---

### Reto 2 — Varios contenedores a la vez

La misma imagen puede arrancar N veces. Cada contenedor es independiente (distinto ID, distinto hostname interno).


In [ ]:
# Limpiar por si ya existen
!docker rm -f iris-1 iris-2 iris-3 2>/dev/null; true

!docker run -d -p 8081:8000 --name iris-1 ml-iris-app
!docker run -d -p 8082:8000 --name iris-2 ml-iris-app
!docker run -d -p 8083:8000 --name iris-3 ml-iris-app

!docker ps --filter "name=iris-" 

In [ ]:
import time, requests
time.sleep(2)

for port in [8081, 8082, 8083]:
    data = requests.get(f"http://127.0.0.1:{port}/info").json()
    print(f"puerto {port}  →  hostname interno: {data['hostname']}  |  en_docker: {data['running_in_docker']}")

Observa cómo **cada contenedor tiene un hostname distinto**, aunque vienen de la MISMA imagen. Son procesos aislados, como tres personas distintas leyendo la misma copia de un libro.

Esto es la base del escalamiento horizontal: si una réplica se cae o se satura, las otras siguen atendiendo.

---

### Reto 3 — Espiar dentro del contenedor

`docker exec` ejecuta comandos dentro de un contenedor que ya está corriendo. Es tu herramienta principal para depurar.


In [ ]:
print("--- ¿qué archivos hay en /app? ---")
!docker exec mi-iris ls -la /app

print("\n--- versión de Python adentro ---")
!docker exec mi-iris python --version

print("\n--- primeras 10 líneas de app.py adentro ---")
!docker exec mi-iris head -n 10 /app/app.py

> **Truco avanzado:** `docker exec -it mi-iris bash` te abre una shell interactiva dentro del contenedor — pero **eso sólo funciona en una terminal real, no en el notebook**. Pruébalo en tu terminal cuando termines.

---

### Reto 4 — Batch de predicciones desde el notebook

Manda 20 flores sintéticas al contenedor y mide cuánto tarda. Así ves cómo una API dockerizada puede servir muchos clientes.


In [ ]:
import time, random, requests

URL = "http://127.0.0.1:8080/predict"
N = 20
random.seed(0)

t0 = time.time()
for i in range(N):
    payload = {
        "sepal_length": round(random.uniform(4, 8), 2),
        "sepal_width":  round(random.uniform(2, 4.5), 2),
        "petal_length": round(random.uniform(1, 7), 2),
        "petal_width":  round(random.uniform(0.1, 2.5), 2),
    }
    r = requests.post(URL, json=payload).json()
    print(f"{i+1:2d}. {payload}  →  {r['class_name']}")
elapsed = time.time() - t0
print(f"\n{N} predicciones en {elapsed:.2f}s  ({N/elapsed:.1f} req/s)")

> **Para reflexionar:** si ese mismo contenedor estuviera en un servidor en la nube en vez de tu laptop, **tu compañero desde otra ciudad podría llamar a la API igual**. No hace falta que instale Python, ni sklearn, ni el modelo. Esa es la esencia del despliegue con Docker.


---

## Parte 5 — Limpieza

Un buen ingeniero siempre limpia. Vamos a detener y eliminar todos los contenedores que creamos.


In [ ]:
# Detener y borrar todos los contenedores de este ejercicio
!docker rm -f mi-iris iris-1 iris-2 iris-3 2>/dev/null; true

# Confirmar que ya no queda ninguno
!docker ps --filter "name=iris" --filter "name=mi-iris"
print("(si la lista de arriba salió vacía — perfecto)")

### (Opcional) Eliminar también la imagen

Si ya no vas a usarla, puedes borrarla para liberar disco. **Hazlo al final del ejercicio, no antes**, o tendrías que reconstruirla.

```bash
docker rmi ml-iris-app
```

---


## Parte 6 — Reflexión y checklist

### Lo que acabas de lograr

1. Entrenaste un modelo y lo **serializaste** a disco.
2. Lo envolviste en una **API** (FastAPI) para que otros sistemas lo consuman.
3. Escribiste un **Dockerfile** que construye una imagen reproducible.
4. Construiste la imagen y la ejecutaste como **contenedor**.
5. Le hablaste al modelo por HTTP desde el notebook.
6. Corriste **tres contenedores en paralelo** de la misma imagen.
7. Espiaste dentro de un contenedor vivo con `docker exec`.
8. Limpiaste todo.

### ¿Qué resolvió Docker concretamente?

| Problema | Sin Docker | Con Docker |
|----------|-----------|-----------|
| "En mi máquina funciona" | Le pides al colega que instale Python, sklearn, FastAPI, con las mismas versiones... | Le mandas la imagen. Punto. |
| Reproducibilidad | El modelo podría predecir distinto con otra versión de sklearn | El contenedor fija **todas** las versiones |
| Escalar | Arrancar N procesos es frágil | `docker run` N veces y listo |
| Onboarding | Medio día de setup para un dev nuevo | 5 minutos: pull + run |

### Checklist de dominio

- [ ] Puedo explicar la diferencia entre **imagen** y **contenedor** con un ejemplo.
- [ ] Sé por qué copiamos `requirements.txt` antes que `app.py` en el Dockerfile.
- [ ] Sé por qué la API usa `host=0.0.0.0` y no `127.0.0.1`.
- [ ] Sé mapear puertos con `-p <local>:<interno>`.
- [ ] Puedo levantar múltiples contenedores de la misma imagen.
- [ ] Puedo ver logs (`docker logs`) y ejecutar comandos dentro (`docker exec`).
- [ ] Puedo detener y limpiar contenedores e imágenes.

### Preguntas para seguir explorando

1. ¿Qué pasa si el modelo pesa 500 MB? ¿Cómo afecta el tamaño de la imagen y su transferencia?
2. ¿Cómo pasarías variables de entorno (ej. el path del modelo, un API key) **sin rehacer la imagen**? Pista: `-e` en `docker run`.
3. Si el contenedor se cae solo, ¿cómo haces que se reinicie automáticamente? Pista: `--restart`.
4. ¿Cómo harías para que varias apps (API + base de datos + MLflow) se levanten juntas? Pista: **docker-compose**.

> Estas preguntas las abordamos en los ejercicios siguientes: `web-service`, `batch-deploy`, y en la sesión de `docker-compose`.

---

**¡Buen trabajo!** Si llegaste hasta acá ejecutando cada celda, ya sabes lo suficiente para dockerizar cualquier modelo que entrenes. Lo demás es variación del mismo patrón.
